In [1]:
# 電子透かしを音声に埋め込む
# 埋め込んだ音声をvcにかける
# 抽出モデルを使って透かしを取り出す

In [2]:
import warnings
import torch
import os
import yaml
warnings.filterwarnings("ignore")
from modules.commons import *
from losses import *

import torchaudio
import librosa

import watermark_hparams as hp

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
def load_model(emb_ckpt_path, extract_ckpt_path, config_path):
    emb_ckpt_path = emb_ckpt_path      # 埋め込みモデルのチェックポイントパス
    extract_ckpt_path = extract_ckpt_path  # 抽出モデルのチェックポイントパス
    config_path = config_path
    config = yaml.safe_load(open(config_path))
    model_params = recursive_munch(config['model_params'])
    watemark_model = build_model(model_params, 'watermarking')
    extracter = build_model(model_params, 'extracter')

    emb_ckpt_params = torch.load(emb_ckpt_path)
    emb_ckpt_params = emb_ckpt_params['net'] if 'net' in emb_ckpt_params else emb_ckpt_params  # adapt to format of self-trained checkpoints

    for key in emb_ckpt_params:
        watemark_model[key].load_state_dict(emb_ckpt_params[key])

    _ = [watemark_model[key].eval() for key in watemark_model]
    _ = [watemark_model[key].to(device) for key in watemark_model]

    extract_ckpt_params = torch.load(extract_ckpt_path)
    extract_ckpt_params = extract_ckpt_params['net'] if 'net' in extract_ckpt_params else extract_ckpt_params  # adapt to format of self-trained checkpoints

    for key in extract_ckpt_params:
        extracter[key].load_state_dict(extract_ckpt_params[key])
    
    _ = [extracter[key].eval() for key in extracter]
    _ = [extracter[key].to(device) for key in extracter]

    return watemark_model, extracter

In [ ]:
# 透かしの埋め込みを実行
emb_ckpt_path = "/workspace/checkpoints/log1/watermark_model_epoch_5_iter_221735.pth"
extract_ckpt_path = "/workspace/checkpoints/log1/extracter_model_epoch_5_iter_221735.pth"
config_path = "/home/FAcodecWatermark/configs/config.yml"
watermark_model, extracter = load_model(emb_ckpt_path, extract_ckpt_path, config_path)

In [ ]:
# 透かしを埋め込む音声の用意
audio_path = "/workspace/Demo/VCTK-corpus/p228/p228_023.wav"
id = audio_path.split("/")[4]
print(id)
source_audio = librosa.load(audio_path, sr=24000)[0]
source_audio = source_audio[:24000 * 30]
source_audio = torch.tensor(source_audio).unsqueeze(0).float().to(device)

In [ ]:
# 透かしの準備
msg = torch.Tensor([0,0,0,0,0,0,0,0,1,1])
msg = msg*2-1
msg = msg.unsqueeze(0).unsqueeze(0)
msg = msg.to(device)

In [ ]:
acc_lst = []

In [ ]:
msg = np.random.choice([0,1], [1, 1, 10])
msg = torch.from_numpy(msg).float()*2 - 1
msg = msg.to(device)
print(msg)

In [ ]:
# 透かしを埋め込む
with torch.no_grad():
    z = watermark_model.encoder(source_audio[None, ...].to(device).float())
    z, quantized, commitment_loss, codebook_loss, timbre, z_c_emb = watermark_model.quantizer(z,
                                                                                            source_audio[None, ...].to(device).float(),
                                                                                            msg,
                                                                                            n_c=2)
    pred_wave = watermark_model.decoder(z)
    path = os.path.join("/workspace/watermark_audio", id)
    os.makedirs(path, exist_ok=True)
    source_name = audio_path.split("/")[-1].split(".")[0]
    save_path = os.path.join(path, f"{source_name}_wm.wav")
    torchaudio.save(save_path, pred_wave[0].cpu(), 24000)

##### 透かしつき音声をVCにかける

In [ ]:
def get_all_data_path(dir_path, extensions=None):
    file_paths = []
    if extensions is not None:
        extensions = tuple(extensions)
    for root, _, files in os.walk(dir_path):
        for file in files:
            if extensions is None or file.endswith(extensions):
                file_path = os.path.join(root, file)
                file_paths.append(file_path)
    return file_paths 

In [ ]:
# 変換された透かしつき音声のパスを確保
file_paths = get_all_data_path("/workspace/converted_audio", extensions=[".wav"])
print(file_paths)

In [ ]:
# 透かしを抽出する
with torch.no_grad():
    print(msg)
    for path in file_paths:
        target = path
        target_audio = librosa.load(target, sr=24000)[0]
        target_audio = target_audio[:24000 * 30]
        target_audio = torch.tensor(target_audio).unsqueeze(0).float().to(device)
        pred_msg = extracter.encoder(target_audio[None, ...].to(device).float())
        acc = [((pred_msg >= 0).eq(msg >= 0).sum().float() / msg.numel()).item()]
        logit_msg = [(pred_msg >= 0).float()]
        print(logit_msg)
        name = os.path.basename(path).split(".")[0]
        print(f"{name} accuracy: {acc[0]*100}%")
        acc_lst.append(acc[0]*100)

In [ ]:
print(len(acc_lst))
print(sum(acc_lst) / len(acc_lst)) # 平均精度を計算